# Spatial Cross-Validation for Temperature Mapping

This notebook loads tiled raster inputs, builds spatial features, trains a `RidgeCV`
regression model under four train/test split strategies, evaluates test predictions,
and creates a combined comparison figure.

**Before running:** update the paths in the configuration cell if your Google Drive
folder differs. Run the notebook from top to bottom.

## 1. Environment setup

Run this cell in Google Colab if the required packages are not already installed.
`torchgeo` and its `RCF` model were present in the original notebook but are not used
by the current workflow, which trains `RidgeCV`; they are therefore omitted here.

In [ ]:
%pip install rasterio scikit-learn scipy matplotlib numpy

## 2. Mount Google Drive

Skip this cell if you are running outside Google Colab or have already mounted Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Imports

In [ ]:
import os
import random

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import numpy as np
import rasterio
from scipy.ndimage import generic_filter
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

## 4. Configuration

The input folders should contain GeoTIFF tiles whose filenames end in row and column
numbers, for example `..._1_1.tif`. The code expects a 12 × 12 tile grid.

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/ADMS_Input_Output"

TARGET_PATH = os.path.join(DATA_ROOT, "Temp_MC_pixel50x50")

INPUT_PATHS = {
    "alb": os.path.join(DATA_ROOT, "alb_pixel_50x50"),
    "evp": os.path.join(DATA_ROOT, "evp_pixel_50x50"),
    "nbv": os.path.join(DATA_ROOT, "nbv_pixel_50x50"),
    "ruf": os.path.join(DATA_ROOT, "ruf_pixel_50x50"),
    "ter": os.path.join(DATA_ROOT, "ter_pixel_50x50"),
    "thc": os.path.join(DATA_ROOT, "thc_pixel_50x50"),
}

GRID_SIZE = 12
TEMPERATURE_MIN = 20
TEMPERATURE_MAX = 25
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 5. Helper functions

- `clean_array`: converts invalid or extreme values to `NaN`.
- `add_spatial_features`: calculates each input raster's local mean and standard deviation.
- `load_tiles`: indexes tile files by their `(row, column)` coordinates.
- `is_bad_tile`: skips unreadable or empty target tiles.

In [ ]:
def clean_array(array):
    """Convert a raster array to float32 and mark invalid/extreme values as NaN."""
    array = array.astype(np.float32, copy=True)
    array[~np.isfinite(array)] = np.nan
    array[(array > 1e5) | (array < -1e5)] = np.nan
    return array


def add_spatial_features(array):
    """Return the original raster, 3x3 local mean, and 3x3 local standard deviation."""
    local_mean = generic_filter(array, np.nanmean, size=3, mode="nearest")
    local_std = generic_filter(array, np.nanstd, size=3, mode="nearest")
    return array, local_mean, local_std


def load_tiles(folder, grid_size=GRID_SIZE):
    """Return {(row, column): filepath} for validly named GeoTIFF tiles."""
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Input folder does not exist: {folder}")

    tiles = {}
    for filename in os.listdir(folder):
        if not filename.lower().endswith(".tif"):
            continue

        name_parts = os.path.splitext(filename)[0].split("_")
        try:
            row, col = int(name_parts[-2]), int(name_parts[-1])
        except (ValueError, IndexError):
            continue

        if 1 <= row <= grid_size and 1 <= col <= grid_size:
            tiles[(row, col)] = os.path.join(folder, filename)

    return tiles


def is_bad_tile(filepath):
    """Return True when a target tile cannot be read or has no usable values."""
    try:
        with rasterio.open(filepath) as src:
            array = src.read(1).astype(np.float32)
        array = array[np.isfinite(array)]
        return array.size == 0 or np.nanmax(array) > 1e10
    except (OSError, ValueError, rasterio.errors.RasterioError):
        return True

## 6. Index the raster tiles and find matching tiles

Only tile coordinates present in the target and all six input folders are retained.
Invalid target tiles are excluded.

In [ ]:
target_tiles = load_tiles(TARGET_PATH)
input_tiles = {
    name: load_tiles(folder)
    for name, folder in INPUT_PATHS.items()
}

valid_keys = []
for key in target_tiles:
    if not all(key in tiles for tiles in input_tiles.values()):
        continue
    if is_bad_tile(target_tiles[key]):
        continue
    valid_keys.append(key)

valid_keys = sorted(valid_keys)

if not valid_keys:
    raise ValueError(
        "No matching valid tiles were found. Check folder paths and tile filenames."
    )

print(f"Clean valid tiles: {len(valid_keys)}")

## 7. Raster reading and feature construction

Each of the six input rasters contributes three feature bands: the original values,
a 3 × 3 local mean, and a 3 × 3 local standard deviation. This gives 18 features
per pixel.

In [ ]:
def read_feature_stack(key):
    """Read all input rasters for one tile and stack their 18 feature bands."""
    feature_layers = []

    for tile_dict in input_tiles.values():
        with rasterio.open(tile_dict[key]) as src:
            array = clean_array(src.read(1))

        original, local_mean, local_std = add_spatial_features(array)
        feature_layers.extend([original, local_mean, local_std])

    return np.stack(feature_layers, axis=-1)


def read_target_tile(key):
    """Read and clean one target temperature tile."""
    with rasterio.open(target_tiles[key]) as src:
        return clean_array(src.read(1))

## 8. Assemble and display the observed temperature map

The first valid target tile determines the dimensions used for the mosaic.

In [ ]:
tile_height, tile_width = read_target_tile(valid_keys[0]).shape
map_shape = (GRID_SIZE * tile_height, GRID_SIZE * tile_width)

true_map = np.full(map_shape, np.nan, dtype=np.float32)

for row, col in valid_keys:
    row_start = (row - 1) * tile_height
    col_start = (col - 1) * tile_width
    true_map[
        row_start:row_start + tile_height,
        col_start:col_start + tile_width,
    ] = read_target_tile((row, col))

plt.figure(figsize=(8, 8))
plt.imshow(
    true_map,
    cmap="turbo",
    vmin=TEMPERATURE_MIN,
    vmax=TEMPERATURE_MAX,
)
plt.title("Original Temperature Map")
plt.colorbar(label="Temperature (°C)")
plt.xlabel("Column (pixels)")
plt.ylabel("Row (pixels)")
plt.tight_layout()
plt.show()

## 9. Train/test split strategies

- `left_right`: tiles in columns 1–6 are used for training.
- `top_bottom`: tiles in rows 1–6 are used for training.
- `outer_inner`: tiles along the outer three rows or columns are used for training.
- `random10`: a random 10% of valid tiles are used for training.

In [ ]:
def get_split(keys, mode):
    """Split tile coordinates into training and testing sets."""
    if mode == "left_right":
        train_keys = [key for key in keys if key[1] <= 6]
    elif mode == "top_bottom":
        train_keys = [key for key in keys if key[0] <= 6]
    elif mode == "outer_inner":
        train_keys = [
            key for key in keys
            if (
                key[0] <= 3 or key[0] >= 10
                or key[1] <= 3 or key[1] >= 10
            )
        ]
    elif mode == "random10":
        sample_size = max(1, int(len(keys) * 0.1))
        train_keys = random.sample(list(keys), sample_size)
    else:
        raise ValueError(f"Unknown split mode: {mode}")

    train_set = set(train_keys)
    test_keys = [key for key in keys if key not in train_set]

    if not train_keys or not test_keys:
        raise ValueError(
            f"Split '{mode}' produced an empty training or testing set."
        )

    return train_keys, test_keys


SPLIT_MODES = [
    "left_right",
    "top_bottom",
    "outer_inner",
    "random10",
]

## 10. Run spatial cross-validation experiments

For each split, the scaler is fitted on training pixels only. Predictions and metrics
are evaluated on the held-out tiles. Pixels with invalid feature values are left as
`NaN` in the prediction map.

In [ ]:
results = {}

for mode in SPLIT_MODES:
    print(f"\n{'=' * 30}\nSplit mode: {mode}\n{'=' * 30}")

    train_keys, test_keys = get_split(valid_keys, mode)

    # Gather valid training pixels from the training tiles.
    X_train_parts = []
    y_train_parts = []

    for key in train_keys:
        X_tile = read_feature_stack(key)
        y_tile = read_target_tile(key)

        valid_mask = (
            np.isfinite(X_tile).all(axis=-1)
            & np.isfinite(y_tile)
            & (y_tile >= 0)
            & (y_tile <= 50)
        )

        if np.any(valid_mask):
            X_train_parts.append(X_tile[valid_mask])
            y_train_parts.append(y_tile[valid_mask])

    if not X_train_parts:
        raise ValueError(
            f"No valid training pixels were found for split '{mode}'."
        )

    X_train = np.vstack(X_train_parts)
    y_train = np.concatenate(y_train_parts)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    regressor = RidgeCV()
    regressor.fit(X_train_scaled, y_train)

    # Build training, prediction, and error maps.
    train_map = np.full_like(true_map, np.nan)
    pred_map = np.full_like(true_map, np.nan)

    for row, col in train_keys:
        row_start = (row - 1) * tile_height
        col_start = (col - 1) * tile_width
        train_map[
            row_start:row_start + tile_height,
            col_start:col_start + tile_width,
        ] = read_target_tile((row, col))

    for key in test_keys:
        row, col = key
        row_start = (row - 1) * tile_height
        col_start = (col - 1) * tile_width

        X_tile = read_feature_stack(key)
        tile_valid_mask = np.isfinite(X_tile).all(axis=-1)

        if not np.any(tile_valid_mask):
            continue

        X_flat = X_tile.reshape(-1, X_tile.shape[-1])
        flat_valid_mask = tile_valid_mask.ravel()

        tile_predictions = np.full(
            X_flat.shape[0], np.nan, dtype=np.float32
        )
        tile_predictions[flat_valid_mask] = regressor.predict(
            scaler.transform(X_flat[flat_valid_mask])
        )
        tile_predictions = tile_predictions.reshape(tile_height, tile_width)

        pred_map[
            row_start:row_start + tile_height,
            col_start:col_start + tile_width,
        ] = tile_predictions

    diff_map = true_map - pred_map

    # Calculate metrics only where both observed and predicted values are finite.
    metric_mask = np.isfinite(true_map) & np.isfinite(pred_map)
    y_true = true_map[metric_mask]
    y_pred = pred_map[metric_mask]

    if y_true.size == 0:
        rmse = np.nan
        r2 = np.nan
        print("No valid test pixels available for metrics.")
    else:
        rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
        denominator = np.sum((y_true - np.mean(y_true)) ** 2)
        r2 = (
            float(1 - np.sum((y_true - y_pred) ** 2) / denominator)
            if denominator > 0 else np.nan
        )
        print(f"Test RMSE: {rmse:.4f}")
        print(f"Test R²:   {r2:.4f}")

    results[mode] = {
        "train_map": train_map,
        "pred_map": pred_map,
        "diff_map": diff_map,
        "rmse": rmse,
        "r2": r2,
    }

## 11. Plot all split results in one figure

Each row corresponds to one split strategy. Columns show the training tiles, observed
test tiles, predictions, and observed-minus-predicted error. Temperature and error
color scales are shared across panels for easier comparison.

In [ ]:
N_COLS = 4

split_modes_plot = [
    "random10",
    "outer_inner",
    "top_bottom",
    "left_right",
]

panel_labels = ["(a)", "(b)", "(c)", "(d)"]
training_titles = [
    f"{panel_labels[0]} Training set (Random 10% split)",
    f"{panel_labels[1]} Training set (Outer–inner split)",
    f"{panel_labels[2]} Training set (Top–bottom split)",
    f"{panel_labels[3]} Training set (Left–right split)",
]

n_modes = len(split_modes_plot)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "font.weight": "bold",
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

fig = plt.figure(figsize=(20, 6.2 * n_modes + 1))
grid_spec = gridspec.GridSpec(
    n_modes + 1,
    N_COLS,
    figure=fig,
    hspace=0.15,
    wspace=0.08,
    height_ratios=[1] * n_modes + [0.05],
)

temperature_norm = mcolors.Normalize(
    vmin=TEMPERATURE_MIN,
    vmax=TEMPERATURE_MAX,
)
temperature_cmap = cm.turbo

finite_errors = [
    results[mode]["diff_map"][np.isfinite(results[mode]["diff_map"])].ravel()
    for mode in split_modes_plot
]
finite_errors = [values for values in finite_errors if values.size > 0]

if finite_errors:
    all_errors = np.concatenate(finite_errors)
    error_limit = float(np.percentile(np.abs(all_errors), 98))
    if error_limit == 0:
        error_limit = 1.0
else:
    error_limit = 1.0

error_norm = mcolors.Normalize(vmin=-error_limit, vmax=error_limit)
error_cmap = cm.bwr

for row, mode in enumerate(split_modes_plot):
    result = results[mode]

    observed_test_map = np.where(
        np.isnan(result["train_map"]) & np.isfinite(true_map),
        true_map,
        np.nan,
    )

    for col in range(N_COLS):
        ax = fig.add_subplot(grid_spec[row, col])

        if col == 0:
            ax.imshow(
                result["train_map"],
                cmap=temperature_cmap,
                norm=temperature_norm,
            )
            ax.set_title(
                training_titles[row],
                fontsize=12,
                fontweight="bold",
                loc="center",
                pad=4,
            )

            # Show the north arrow on the final training panel only.
            if row == n_modes - 1:
                ax.annotate(
                    "",
                    xy=(0.90, 0.88),
                    xytext=(0.90, 0.72),
                    xycoords="axes fraction",
                    arrowprops={
                        "facecolor": "black",
                        "edgecolor": "black",
                        "width": 1.5,
                        "headwidth": 7,
                        "headlength": 8,
                    },
                    zorder=20,
                )
                ax.text(
                    0.90, 0.91, "North",
                    transform=ax.transAxes,
                    ha="center",
                    va="bottom",
                    fontsize=12,
                    fontweight="bold",
                    color="black",
                    zorder=20,
                )

        elif col == 1:
            ax.imshow(
                observed_test_map,
                cmap=temperature_cmap,
                norm=temperature_norm,
            )
            ax.set_title("Observed test set", fontsize=12, fontweight="bold", pad=6)

        elif col == 2:
            ax.imshow(
                result["pred_map"],
                cmap=temperature_cmap,
                norm=temperature_norm,
            )
            ax.set_title(
                f"Prediction (Test R² = {result['r2']:.3f})",
                fontsize=12,
                fontweight="bold",
                pad=6,
            )

        else:
            ax.imshow(
                result["diff_map"],
                cmap=error_cmap,
                norm=error_norm,
            )
            ax.set_title(
                "Error (Observed − Prediction)",
                fontsize=12,
                fontweight="bold",
                pad=6,
            )

        ax.tick_params(axis="both", labelsize=12)

        if col > 0:
            ax.tick_params(axis="y", left=True, labelleft=False)

        ax.set_xlabel(
            "Column" if row == n_modes - 1 else "",
            fontsize=12,
            fontweight="bold",
        )
        ax.set_ylabel(
            "Row" if col == 0 else "",
            fontsize=12,
            fontweight="bold",
        )

# Temperature colorbar spans the first three columns.
temp_colorbar_ax = fig.add_subplot(grid_spec[n_modes, :3])
temp_colorbar = fig.colorbar(
    cm.ScalarMappable(norm=temperature_norm, cmap=temperature_cmap),
    cax=temp_colorbar_ax,
    orientation="horizontal",
)
temp_colorbar.set_label("Temperature (°C)", fontsize=12, fontweight="bold")
temp_colorbar.ax.tick_params(labelsize=12)

# Error colorbar occupies the final column.
error_colorbar_ax = fig.add_subplot(grid_spec[n_modes, 3])
error_colorbar = fig.colorbar(
    cm.ScalarMappable(norm=error_norm, cmap=error_cmap),
    cax=error_colorbar_ax,
    orientation="horizontal",
)
error_colorbar.set_label("Error (°C)", fontsize=12, fontweight="bold")
error_colorbar.ax.tick_params(labelsize=12)

fig.savefig("cv_results.png", dpi=300, bbox_inches="tight")
plt.show()

## 12. Output

The combined figure is saved as `cv_results.png` in the current working directory.
In Google Colab, you can download it from the Files panel or copy it to Google Drive
if you want to keep it permanently.